Hands-on Task 2: Build a Review Pipeline with Claude 

Objective 

Create a multi-step LLM pipeline with sequential processing. 

Task 

Define a state structure: 

class ReviewState(TypedDict): 

    product: str 

    review: str 

    sentiment: str 

    reply: str 

`` 

Build the following pipeline: 

START → review_node → sentiment_node → reply_node → END 

Node Requirements 

1. review_node 

Generate a short product review using Claude 

2. sentiment_node 

Classify the review into:  

Positive 

Negative 

Neutral 

3. reply_node 

Generate a one-line brand response based on the sentiment 

Test Input 

product = "wireless noise-cancelling headphones" 

Skills Tested 

Chaining multiple LLM steps 

State management across nodes 

Prompt design for generation and classification 

Using Claude in graph-based workflows 

In [9]:
from typing import TypedDict
 
from langgraph.graph import StateGraph, START, END
 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
 
from langchain_anthropic import ChatAnthropic

from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END

import os

In [10]:
#Step 2: Create the LLM
 
# Load ANTHROPIC_API_KEY from .env
load_dotenv(override=True)
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if not anthropic_api_key:
    raise ValueError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

# ── Initialise Claude chat model ──────────────────────────────────────────
llm = ChatAnthropic(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT"),
    temperature=0
)
print("Setup complete!")

Setup complete!


In [12]:
#Step 3: Create the State
class ReviewState(TypedDict):
    product: str
    review: str
    sentiment: str
    reply: str

In [14]:
#Step 4: Create Builder
builder = StateGraph(ReviewState)

In [15]:
#Step 5: Review Prompt
review_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Write a short product review for the given product."
    ),
    (
        "human",
        "{product}"
    )
])

In [16]:
#Step 6: Review Chain
review_chain = review_prompt | llm | StrOutputParser()

In [17]:
#Step 7: Review Node
def review_node(state):
    review = review_chain.invoke({
        "product": state["product"]
    })
 
    state["review"] = review
    return state

In [18]:
#Step 8: Sentiment Prompt
sentiment_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Classify the review as Positive, Negative or Neutral. Reply with only one word."
    ),
    (
        "human",
        "{review}"
    )
])

In [19]:
#Step 9: Sentiment Chain
sentiment_chain = sentiment_prompt | llm | StrOutputParser()

In [20]:
#Step 10: Sentiment Node
def sentiment_node(state):
 
    sentiment = sentiment_chain.invoke({
        "review": state["review"]
    })
 
    state["sentiment"] = sentiment
 
    return state

In [21]:
#Step 11: Reply Prompt
reply_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Generate a one-line brand reply based on the sentiment.
 
Positive → Thank the customer.
 
Negative → Apologize politely.
 
Neutral → Appreciate the feedback.
"""
    ),
    (
        "human",
        "{sentiment}"
    )
])

In [23]:
#Step 12: Reply Chain
reply_chain = reply_prompt | llm | StrOutputParser()

In [24]:
#Step 13: Reply Node
def reply_node(state):
 
    reply = reply_chain.invoke({
        "sentiment": state["sentiment"]
    })
 
    state["reply"] = reply
 
    return state

In [25]:
#Step 14: Add Nodes
builder.add_node("review_node", review_node)
builder.add_node("sentiment_node", sentiment_node)
builder.add_node("reply_node", reply_node)
#Step 15: Add Edges
builder.add_edge(START, "review_node")
builder.add_edge("review_node", "sentiment_node")
builder.add_edge("sentiment_node", "reply_node")
builder.add_edge("reply_node", END)

In [27]:
#Step 16: Compile
graph = builder.compile()

In [28]:
#Step 17: Test
result = graph.invoke({
    "product": "Wireless Noise Cancelling Headphones"
})

In [31]:
#Step 18: Print Output
print("=" * 50)

print("Product:")
print(result["product"])
 
print("\nReview:")
print(result["review"])
 
print("\nSentiment:")
print(result["sentiment"])
 
print("\nReply:")
print(result["reply"])

Product:
Wireless Noise Cancelling Headphones

Review:
# Wireless Noise Cancelling Headphones - Review

**Rating: 4.5/5 Stars**

These headphones deliver impressive performance for the price. The noise cancellation effectively blocks out ambient sounds, making them ideal for commutes, flights, or focused work sessions. The wireless connectivity is stable, and the battery life easily lasts through a full day of use.

**Pros:**
- Excellent active noise cancellation
- Comfortable fit for extended wear
- Good sound quality with balanced audio
- Long battery life (25+ hours)
- Intuitive touch controls

**Cons:**
- Slightly bulky for travel
- Noise cancellation could be stronger in very loud environments
- App connectivity occasionally glitchy

**Verdict:**
A solid choice for anyone seeking quality noise cancellation without breaking the bank. Perfect for travelers and remote workers. Highly recommended.

Sentiment:
Positive

Reply:
Thank you so much for your kind words—we truly appreciate y